In [5]:
from collections.abc import Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Literal, Protocol

import joblib
from pydantic import BaseModel, ConfigDict, Field, field_validator

ARTIFACT_SCHEMA_VERSION = "wine-quality-bundle-v1"
PREPROCESSING_VERSION = "wine-red-features-v1"

MANIFEST_FILENAME = "manifest.json"
MODEL_FILENAME = "model.joblib"

FEATURE_NAMES = (
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol",
)

QualityBand = Literal["needs_review", "acceptable", "excellent"]

OUTPUT_LABELS = ("needs_review", "acceptable", "excellent")

print("Constantes definidas correctamente")

Constantes definidas correctamente


In [6]:
class WineQualityEstimator(Protocol):
    def predict(self, features: list[list[float]]) -> Sequence[str]: ...

    def predict_proba(
        self, features: list[list[float]]
    ) -> Sequence[Sequence[float]]: ...


print("Interfaz del estimador definida")

Interfaz del estimador definida


In [7]:
class ArtifactManifest(BaseModel):
    model_config = ConfigDict(extra="forbid")

    schema_version: str
    model_version: str = Field(min_length=1)
    preprocessing_version: str
    feature_names: tuple[str, ...]
    output_labels: tuple[QualityBand, ...]
    estimator_type: str = Field(min_length=1)

    @field_validator("schema_version")
    @classmethod
    def validar_schema(cls, value):
        if value != ARTIFACT_SCHEMA_VERSION:
            raise ValueError("Versión del esquema incorrecta")
        return value

    @field_validator("preprocessing_version")
    @classmethod
    def validar_preprocesado(cls, value):
        if value != PREPROCESSING_VERSION:
            raise ValueError("Versión del preprocesado incorrecta")
        return value

    @field_validator("feature_names")
    @classmethod
    def validar_features(cls, value):
        if value != FEATURE_NAMES:
            raise ValueError("Las features no coinciden")
        return value

    @field_validator("output_labels")
    @classmethod
    def validar_etiquetas(cls, value):
        if value != OUTPUT_LABELS:
            raise ValueError("Las etiquetas no coinciden")
        return value


print("ArtifactManifest definido")

ArtifactManifest definido


In [8]:
from sklearn.dummy import DummyClassifier

X = [
    [7.4, 0.70, 0.00, 1.9, 0.076, 11, 34, 0.9978, 3.51, 0.56, 9.4],
    [7.8, 0.88, 0.00, 2.6, 0.098, 25, 67, 0.9968, 3.20, 0.68, 9.8],
    [8.1, 0.28, 0.40, 2.0, 0.050, 15, 30, 0.9950, 3.30, 0.80, 12.5],
]

y = ["needs_review", "acceptable", "excellent"]

estimator = DummyClassifier(strategy="most_frequent")
estimator.fit(X, y)

print("Modelo creado:", type(estimator).__name__)
print("Etiquetas:", estimator.classes_)
print("Predicción de ejemplo:", estimator.predict([X[0]]))

Modelo creado: DummyClassifier
Etiquetas: ['acceptable' 'excellent' 'needs_review']
Predicción de ejemplo: ['acceptable']


In [9]:
@dataclass(frozen=True)
class LoadedModelBundle:
    estimator: WineQualityEstimator
    manifest: ArtifactManifest


print("LoadedModelBundle definido")

LoadedModelBundle definido


In [10]:
def validar_estimador(estimator):

    if not callable(getattr(estimator, "predict", None)):
        raise ValueError("El modelo no tiene predict")

    if not callable(getattr(estimator, "predict_proba", None)):
        raise ValueError("El modelo no tiene predict_proba")

    print("El estimador tiene predict y predict_proba")

In [11]:
validar_estimador(estimator)

El estimador tiene predict y predict_proba


In [12]:
def create_manifest(estimator, model_version):

    validar_estimador(estimator)

    tipo_modelo = type(estimator).__name__

    manifest = ArtifactManifest(
        schema_version=ARTIFACT_SCHEMA_VERSION,
        model_version=model_version,
        preprocessing_version=PREPROCESSING_VERSION,
        feature_names=FEATURE_NAMES,
        output_labels=OUTPUT_LABELS,
        estimator_type=tipo_modelo,
    )

    return manifest

In [13]:
manifest = create_manifest(estimator, "wine-quality-demo-v1")

print("Objeto creado:")
print(manifest)

print("\nManifiesto en JSON:")
print(manifest.model_dump_json(indent=2))

El estimador tiene predict y predict_proba
Objeto creado:
schema_version='wine-quality-bundle-v1' model_version='wine-quality-demo-v1' preprocessing_version='wine-red-features-v1' feature_names=('fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol') output_labels=('needs_review', 'acceptable', 'excellent') estimator_type='DummyClassifier'

Manifiesto en JSON:
{
  "schema_version": "wine-quality-bundle-v1",
  "model_version": "wine-quality-demo-v1",
  "preprocessing_version": "wine-red-features-v1",
  "feature_names": [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol"
  ],
  "output_labels": [
    "needs_review",
    "acceptable",
    "excellent"
  ],
  "estimator_type": "DummyClassifier"
}


In [14]:
def save_model_bundle(bundle_path, estimator, manifest=None):

    # Comprobar el estimador
    validar_estimador(estimator)

    # Crear el manifiesto si no existe
    if manifest is None:
        manifest = create_manifest(estimator, "unknown")

    # Comprobar las features
    if manifest.feature_names != FEATURE_NAMES:
        raise ValueError("Las features no coinciden")

    # Comprobar el tipo
    if manifest.estimator_type != type(estimator).__name__:
        raise ValueError("El tipo de estimador no coincide")

    # Crear la carpeta
    bundle_path = Path(bundle_path)
    bundle_path.mkdir(parents=True, exist_ok=True)

    # Guardar el modelo
    model_path = bundle_path / MODEL_FILENAME
    joblib.dump({"estimator": estimator}, model_path, compress=3)

    # Guardar el manifiesto
    manifest_path = bundle_path / MANIFEST_FILENAME
    manifest_path.write_text(manifest.model_dump_json(indent=2), encoding="utf-8")

    return manifest

In [15]:
bundle_path = Path("prueba_bundle")

manifest_guardado = save_model_bundle(bundle_path, estimator, manifest)

print("\nArchivos creados:")

for archivo in bundle_path.iterdir():
    print(archivo.name, "-", archivo.stat().st_size, "bytes")

El estimador tiene predict y predict_proba

Archivos creados:
manifest.json - 524 bytes
model.joblib - 474 bytes


In [16]:
manifest_path = bundle_path / "manifest.json"

print(manifest_path.read_text(encoding="utf-8"))

{
  "schema_version": "wine-quality-bundle-v1",
  "model_version": "wine-quality-demo-v1",
  "preprocessing_version": "wine-red-features-v1",
  "feature_names": [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol"
  ],
  "output_labels": [
    "needs_review",
    "acceptable",
    "excellent"
  ],
  "estimator_type": "DummyClassifier"
}


In [17]:
def load_model_bundle(bundle_path):

    bundle_path = Path(bundle_path)

    manifest_path = bundle_path / MANIFEST_FILENAME
    model_path = bundle_path / MODEL_FILENAME

    if not manifest_path.is_file() or not model_path.is_file():
        raise FileNotFoundError("Falta manifest.json o model.joblib")

    # Primero validar el manifiesto
    manifest = ArtifactManifest.model_validate_json(
        manifest_path.read_text(encoding="utf-8")
    )

    # Después cargar el estimador
    payload = joblib.load(model_path)

    if not isinstance(payload, dict):
        raise ValueError("El archivo no contiene un diccionario")

    estimator = payload.get("estimator")

    validar_estimador(estimator)

    if manifest.estimator_type != type(estimator).__name__:
        raise ValueError("El tipo de estimador no coincide")

    return LoadedModelBundle(estimator, manifest)

In [18]:
bundle = load_model_bundle(bundle_path)

print("Modelo cargado:", type(bundle.estimator).__name__)
print("Versión:", bundle.manifest.model_version)
print("Features:", bundle.manifest.feature_names)
print("Etiquetas:", bundle.manifest.output_labels)

El estimador tiene predict y predict_proba
Modelo cargado: DummyClassifier
Versión: wine-quality-demo-v1
Features: ('fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol')
Etiquetas: ('needs_review', 'acceptable', 'excellent')


In [19]:
from pydantic import FiniteFloat


class WineQualityRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")

    fixed_acidity: float = Field(ge=0, le=20)
    volatile_acidity: float = Field(ge=0, le=2)
    citric_acid: float = Field(ge=0, le=2)
    residual_sugar: float = Field(ge=0, le=20)
    chlorides: float = Field(ge=0, le=1)
    free_sulfur_dioxide: float = Field(ge=0, le=100)
    total_sulfur_dioxide: float = Field(ge=0, le=300)
    density: float = Field(ge=0.98, le=1.01)
    ph: float = Field(ge=2.5, le=4.5)
    sulphates: float = Field(ge=0, le=3)
    alcohol: float = Field(ge=5, le=20)


class WineQualityPrediction(BaseModel):
    model_config = ConfigDict(extra="forbid")

    quality_band: QualityBand
    confidence: FiniteFloat = Field(ge=0, le=1)
    model_version: str = Field(min_length=1)
    preprocessing_version: str = Field(min_length=1)


print("Contratos de entrada y salida definidos")

Contratos de entrada y salida definidos


In [20]:
try:
    request_invalido = WineQualityRequest(
        fixed_acidity=7.4,
        volatile_acidity=0.70,
        citric_acid=0.00,
        residual_sugar=1.9,
        chlorides=0.076,
        free_sulfur_dioxide=11,
        total_sulfur_dioxide=34,
        density=0.9978,
        ph=99,
        sulphates=0.56,
        alcohol=9.4,
    )

except Exception as error:
    print("Error detectado:")
    print(error)

Error detectado:
1 validation error for WineQualityRequest
ph
  Input should be less than or equal to 4.5 [type=less_than_equal, input_value=99, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


In [21]:
try:
    manifest_invalido = ArtifactManifest(
        schema_version="version_incorrecta",
        model_version="wine-quality-demo-v1",
        preprocessing_version=PREPROCESSING_VERSION,
        feature_names=FEATURE_NAMES,
        output_labels=OUTPUT_LABELS,
        estimator_type="DummyClassifier",
    )

except Exception as error:
    print("Error detectado:")
    print(error)

Error detectado:
1 validation error for ArtifactManifest
schema_version
  Value error, Versión del esquema incorrecta [type=value_error, input_value='version_incorrecta', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


In [22]:
import csv
from pathlib import Path

import pandas as pd

In [23]:
def predict_file(input_path, output_path, bundle_path):
    """Lee el CSV, predice y guarda los resultados."""

    # Cargar el bundle
    bundle = load_model_bundle(bundle_path)

    # Leer el CSV
    with input_path.open("r", encoding="utf-8", newline="") as file:
        reader = csv.DictReader(file)
        rows = list(reader)

    resultados = []

    # Predecir todas las filas
    for row in rows:
        sample_id = row.pop("sample_id")

        request = WineQualityRequest(**row)

        prediction = infer_wine_quality(bundle, request)

        resultados.append(
            {
                "sample_id": sample_id,
                "predicted_class": prediction.quality_band,
                "confidence": prediction.confidence,
                "model_version": prediction.model_version,
                "preprocessing_version": prediction.preprocessing_version,
            }
        )

    # Guardar los resultados
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with output_path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=[
                "sample_id",
                "predicted_class",
                "confidence",
                "model_version",
                "preprocessing_version",
            ],
        )

        writer.writeheader()
        writer.writerows(resultados)

    print(f"Predicciones guardadas: {len(resultados)}")

    return 0

In [24]:
input_path = Path("vinos_entrada.csv")
output_path = Path("vinos_predicciones.csv")
bundle_path = Path("prueba_bundle")

# Crear un DataFrame con los vinos
df = pd.DataFrame(X, columns=FEATURE_NAMES)

# Añadir un identificador
df.insert(0, "sample_id", ["red-001", "red-002", "red-003"])

# Guardar el CSV
df.to_csv(input_path, index=False)

# Mostrar los datos
display(df)

,sample_id,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,ph,sulphates,alcohol
0,red-001,7.4,0.70,0.0,1.9,0.076,11,34,0.9978,3.51,0.56,9.4
1,red-002,7.8,0.88,0.0,2.6,0.098,25,67,0.9968,3.20,0.68,9.8
2,red-003,8.1,0.28,0.4,2.0,0.050,15,30,0.9950,3.30,0.80,12.5


In [28]:
from dataclasses import dataclass


@dataclass
class WineFeatures:
    fixed_acidity: float
    volatile_acidity: float
    citric_acid: float
    residual_sugar: float
    chlorides: float
    free_sulfur_dioxide: float
    total_sulfur_dioxide: float
    density: float
    ph: float
    sulphates: float
    alcohol: float

    def as_vector(self):
        return [
            self.fixed_acidity,
            self.volatile_acidity,
            self.citric_acid,
            self.residual_sugar,
            self.chlorides,
            self.free_sulfur_dioxide,
            self.total_sulfur_dioxide,
            self.density,
            self.ph,
            self.sulphates,
            self.alcohol,
        ]


def preprocess_wine_request(request):
    return WineFeatures(**request.model_dump())

In [26]:
def infer_wine_quality(bundle, request):

    # Preprocesar los datos
    features = preprocess_wine_request(request)
    matrix = [features.as_vector()]

    # Hacer la predicción
    etiquetas = bundle.estimator.predict(matrix)
    probabilidades = bundle.estimator.predict_proba(matrix)

    # Comprobar los resultados
    if len(etiquetas) != 1 or len(probabilidades) != 1:
        raise ValueError("El modelo debe devolver una predicción por fila")

    if len(probabilidades[0]) == 0:
        raise ValueError("El modelo no devolvió probabilidades")

    # Obtener la etiqueta y confianza
    etiqueta = str(etiquetas[0])
    confianza = max(float(p) for p in probabilidades[0])

    # Crear la respuesta
    resultado = WineQualityPrediction(
        quality_band=etiqueta,
        confidence=confianza,
        model_version=bundle.manifest.model_version,
        preprocessing_version=bundle.manifest.preprocessing_version,
    )

    return resultado

In [29]:
resultado = predict_file(
    input_path=input_path,
    output_path=output_path,
    bundle_path=bundle_path,
)

print("Código de salida:", resultado)
print("¿Se ha creado el CSV?", output_path.exists())

El estimador tiene predict y predict_proba
Predicciones guardadas: 3
Código de salida: 0
¿Se ha creado el CSV? True


In [30]:
df_predicciones = pd.read_csv(output_path)

display(df_predicciones)

,sample_id,predicted_class,confidence,model_version,preprocessing_version
0,red-001,acceptable,1.0,wine-quality-demo-v1,wine-red-features-v1
1,red-002,acceptable,1.0,wine-quality-demo-v1,wine-red-features-v1
2,red-003,acceptable,1.0,wine-quality-demo-v1,wine-red-features-v1


In [31]:
# Copiar el DataFrame original
df_invalido = df.copy()

# Introducir un pH inválido en la segunda fila
df_invalido.loc[1, "ph"] = 99

# Guardar otro CSV de entrada
input_invalido = Path("vinos_invalidos.csv")
output_invalido = Path("predicciones_invalidas.csv")

df_invalido.to_csv(input_invalido, index=False)

display(df_invalido)

,sample_id,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,ph,sulphates,alcohol
0,red-001,7.4,0.70,0.0,1.9,0.076,11,34,0.9978,3.51,0.56,9.4
1,red-002,7.8,0.88,0.0,2.6,0.098,25,67,0.9968,99.00,0.68,9.8
2,red-003,8.1,0.28,0.4,2.0,0.050,15,30,0.9950,3.30,0.80,12.5


In [32]:
# Eliminar una salida anterior, si existiese, para comprobar
# que no se publica un nuevo CSV tras el error.
output_invalido.unlink(missing_ok=True)

try:
    predict_file(
        input_path=input_invalido,
        output_path=output_invalido,
        bundle_path=bundle_path,
    )

except Exception as error:
    print("Error detectado:")
    print(error)

print("\n¿Existe el CSV de salida?", output_invalido.exists())

El estimador tiene predict y predict_proba
Error detectado:
1 validation error for WineQualityRequest
ph
  Input should be less than or equal to 4.5 [type=less_than_equal, input_value='99.0', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal

¿Existe el CSV de salida? False
